# 64×64 Streaming 2D Convolution on PYNQ 
This notebook tests the `conv2d_stream` HLS accelerator for a **64×64 image** and a **3×3 kernel** using an `ap_fixed<32,16>` data type.

The accelerator performs full convolution using a streaming AXI-DMA input/output path. Kernel coefficients are written once through AXI-Lite registers before streaming begins.

Expected dimensions:

```text
Input image      : 64 × 64
Kernel           : 3 × 3
Output image     : 66 × 66
DMA input words  : 4224 = 66 × 64 = 4096 real pixels + 128 trailing zeros
DMA output words : 4356 = 66 × 66
Data format      : ap_fixed<32,16> packed into 32-bit AXI words
```

The HLS implementation assumes the kernel is already in the sliding-window orientation. If mathematical convolution is desired, provide the already-flipped kernel.

Important: for this `ap_fixed<32,16>` version, the DMA buffers contain **raw 32-bit fixed-point bit patterns**, not normal integer or float values.


## 1. Imports and overlay loading

Place `conv2d_design.bit` and `conv2d_design.hwh` in the same directory as this notebook before running it.


In [1]:
import time
import numpy as np
from pynq import Overlay, allocate


In [2]:
ol = Overlay("conv2d-fixed.bit")
print("Overlay loaded successfully")
print("IP blocks:")
for name in ol.ip_dict.keys():
    print("  ", name)

# Expected handles from the Vivado block design
dma = ol.axi_dma_0
conv2d = ol.conv2d_stream_0
hw_timer = ol.axi_timer_0


Overlay loaded successfully
IP blocks:
   conv2d_stream_0
   axi_timer_0
   axi_dma_0
   ps7


## 2. Dimensions

These constants must match the HLS header file.


In [3]:
IH = 64
IW = 64
KH = 3
KW = 3

OH = IH + KH - 1   # 66
OW = IW + KW - 1   # 66

N_INPUT_REAL = IH * IW          # 4096 real image pixels
N_INPUT_STREAM = OH * IW        # 66 * 64 = 4224, because current HLS reads two extra zero rows
N_OUTPUT = OH * OW              # 66 * 66 = 4356

# Must match HLS typedef: ap_fixed<32,16>
FIXED_TOTAL_BITS = 32
FIXED_INT_BITS = 16
FIXED_FRAC_BITS = FIXED_TOTAL_BITS - FIXED_INT_BITS
FIXED_SCALE = 1 << FIXED_FRAC_BITS

print(f"Input image shape     : {IH} x {IW}")
print(f"Kernel shape          : {KH} x {KW}")
print(f"Output image shape    : {OH} x {OW}")
print(f"DMA input words       : {N_INPUT_STREAM}")
print(f"DMA output words      : {N_OUTPUT}")
print(f"Fixed-point format    : ap_fixed<{FIXED_TOTAL_BITS},{FIXED_INT_BITS}>")
print(f"Fractional bits       : {FIXED_FRAC_BITS}")


Input image shape     : 64 x 64
Kernel shape          : 3 x 3
Output image shape    : 66 x 66
DMA input words       : 4224
DMA output words      : 4356
Fixed-point format    : ap_fixed<32,16>
Fractional bits       : 16


## 3. AXI-Lite register map

These offsets are the typical Vitis HLS AXI-Lite register offsets for this function signature. Confirm against the generated HLS driver header if needed:

```bash
grep -n "k00\|k01\|k02\|k10\|k11\|k12\|k20\|k21\|k22" \
  hls_2dconv_stream_pipelined/sol1/impl/ip/drivers/conv2d_stream_v1_0/src/xconv2d_stream_hw.h
```


In [4]:
CTRL_REG = 0x00

K_OFFSETS = {
    "k00": 0x10,
    "k01": 0x18,
    "k02": 0x20,
    "k10": 0x28,
    "k11": 0x30,
    "k12": 0x38,
    "k20": 0x40,
    "k21": 0x48,
    "k22": 0x50,
}

def float_to_fixed_u32(x):
    """
    Convert Python float/int values to raw unsigned 32-bit two's-complement
    bit patterns matching ap_fixed<32,16>.
    """
    arr = np.asarray(x, dtype=np.float64)
    scaled = np.round(arr * FIXED_SCALE).astype(np.int64)
    wrapped = scaled & 0xFFFFFFFF
    return wrapped.astype(np.uint32)

def fixed_u32_to_float(x):
    """
    Convert raw unsigned 32-bit fixed-point bit patterns from DMA/MMIO
    back to float values for inspection and comparison.
    """
    arr = np.asarray(x, dtype=np.uint32)
    signed = arr.view(np.int32).astype(np.int64)
    return signed.astype(np.float64) / FIXED_SCALE

def fixed_u32_to_int_if_exact(x):
    """Convenience display helper for integer-valued fixed-point outputs."""
    vals = fixed_u32_to_float(x)
    rounded = np.rint(vals).astype(np.int64)
    if np.allclose(vals, rounded, atol=1.0 / FIXED_SCALE):
        return rounded
    return vals

def write_kernel(ip, kernel):
    """Write a 3x3 ap_fixed<32,16> kernel to AXI-Lite registers."""
    kernel = np.asarray(kernel, dtype=np.float64)
    assert kernel.shape == (3, 3)

    k_raw = float_to_fixed_u32(kernel)

    ip.write(K_OFFSETS["k00"], int(k_raw[0, 0]))
    ip.write(K_OFFSETS["k01"], int(k_raw[0, 1]))
    ip.write(K_OFFSETS["k02"], int(k_raw[0, 2]))

    ip.write(K_OFFSETS["k10"], int(k_raw[1, 0]))
    ip.write(K_OFFSETS["k11"], int(k_raw[1, 1]))
    ip.write(K_OFFSETS["k12"], int(k_raw[1, 2]))

    ip.write(K_OFFSETS["k20"], int(k_raw[2, 0]))
    ip.write(K_OFFSETS["k21"], int(k_raw[2, 1]))
    ip.write(K_OFFSETS["k22"], int(k_raw[2, 2]))

def start_ip(ip):
    """Start the HLS IP."""
    ip.write(CTRL_REG, 0x01)

def wait_ip_done(ip, timeout_s=1.0):
    """Optional polling of ap_done. DMA waits are usually enough."""
    t0 = time.perf_counter()
    while (ip.read(CTRL_REG) & 0x02) == 0:
        if time.perf_counter() - t0 > timeout_s:
            raise TimeoutError("conv2d_stream did not assert ap_done")
    return True


## 4. Python reference model

This matches the HLS implementation: the kernel is already in the orientation used by the sliding-window hardware, so the reference does **not** flip the kernel again.


In [5]:
def conv2d_full_preflipped_kernel(image, kernel):
    """
    Reference model matching the HLS implementation.

    The kernel is assumed to already be in the orientation used by
    the sliding-window hardware. This function does not flip it again.

    This reference computes in float64. For the integer-valued test below,
    ap_fixed<32,16> should match exactly as long as the output remains inside
    the fixed-point range.
    """
    image = np.asarray(image, dtype=np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)

    ih, iw = image.shape
    kh, kw = kernel.shape
    oh = ih + kh - 1
    ow = iw + kw - 1

    # For a 3x3 kernel, the HLS line buffer convention corresponds to a
    # 2-pixel zero border around the image in this reference model.
    padded = np.zeros((oh + 2, ow + 2), dtype=np.float64)
    padded[2:2+ih, 2:2+iw] = image

    out = np.zeros((oh, ow), dtype=np.float64)
    for r in range(oh):
        for c in range(ow):
            window = padded[r:r+kh, c:c+kw]
            out[r, c] = np.sum(window * kernel)

    return out


## 5. Prepare input image, kernel, and expected output

To keep the notebook readable, only small slices of the 64×64 image and 66×66 output are printed.


In [6]:
# Deterministic 64x64 test image.
# Keep values within a small range so ap_fixed<32,16> does not overflow.
# With this image and kernel, output magnitudes remain well below ±32768.
image = (np.arange(1, IH * IW + 1, dtype=np.int32).reshape(IH, IW) % 256).astype(np.float64)

# Kernel loaded into HLS. This is assumed to be in sliding-window orientation.
kernel = np.array([
    [-1, -2, -3],
    [-4, -5, -6],
    [-7, -8, -9],
], dtype=np.float64)

expected = conv2d_full_preflipped_kernel(image, kernel)

print("Input image shape:", image.shape)
print("Top-left 5x5 of input image:")
print(image[:5, :5].astype(int))

print("Kernel loaded into HLS:")
print(kernel)

print("Expected output shape:", expected.shape)
print("Top-left 8x8 of expected output:")
print(expected[:8, :8].astype(int))
print("Expected output range:", expected.min(), "to", expected.max())


Input image shape: (64, 64)
Top-left 5x5 of input image:
[[  1   2   3   4   5]
 [ 65  66  67  68  69]
 [129 130 131 132 133]
 [193 194 195 196 197]
 [  1   2   3   4   5]]
Kernel loaded into HLS:
[[-1. -2. -3.]
 [-4. -5. -6.]
 [-7. -8. -9.]]
Expected output shape: (66, 66)
Top-left 8x8 of expected output:
[[   -9   -26   -50   -74   -98  -122  -146  -170]
 [ -591 -1131 -1618 -1657 -1696 -1735 -1774 -1813]
 [-1554 -2931 -4128 -4173 -4218 -4263 -4308 -4353]
 [-2706 -5043 -7008 -7053 -7098 -7143 -7188 -7233]
 [-1554 -2803 -3744 -3789 -3834 -3879 -3924 -3969]
 [-1170 -2099 -2784 -2829 -2874 -2919 -2964 -3009]
 [-1554 -2931 -4128 -4173 -4218 -4263 -4308 -4353]
 [-2706 -5043 -7008 -7053 -7098 -7143 -7188 -7233]]
Expected output range: -9708.0 to 0.0


## 6. Allocate DMA buffers

The current HLS implementation reads `OH × IW = 4224` input words. The first 4096 words are the real image pixels, and the final 128 words are zeros representing two extra zero rows.

For the `ap_fixed<32,16>` version, the DMA buffers use `np.uint32` because each element is a raw fixed-point bit pattern.


In [7]:
in_buf = allocate(shape=(N_INPUT_STREAM,), dtype=np.uint32)
out_buf = allocate(shape=(N_OUTPUT,), dtype=np.uint32)

image_raw = float_to_fixed_u32(image.flatten())
zero_raw = np.uint32(0)

in_buf[:N_INPUT_REAL] = image_raw
in_buf[N_INPUT_REAL:] = zero_raw
out_buf[:] = 0

print(f"Input buffer shape : {in_buf.shape}")
print(f"Output buffer shape: {out_buf.shape}")
print("First 8 input words as float:", fixed_u32_to_float(np.array(in_buf[:8])))
print("Last 8 input words, should be zeros:", fixed_u32_to_float(np.array(in_buf[-8:])))


Input buffer shape : (4224,)
Output buffer shape: (4356,)
First 8 input words as float: [1. 2. 3. 4. 5. 6. 7. 8.]
Last 8 input words, should be zeros: [0. 0. 0. 0. 0. 0. 0. 0.]


## 7. Run the accelerator using AXI DMA

Safe order:

1. Program kernel coefficients over AXI-Lite
2. Start DMA receive channel
3. Start DMA send channel
4. Start the HLS IP
5. Wait for both DMA channels


In [8]:
print("--- DMA Streaming 2D Convolution, ap_fixed<32,16> ---")

write_kernel(conv2d, kernel)

t_start = time.perf_counter()

dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)
start_ip(conv2d)

dma.sendchannel.wait()
dma.recvchannel.wait()

t_dma = time.perf_counter() - t_start

hw_out = fixed_u32_to_float(np.array(out_buf)).reshape(OH, OW)

print(f"DMA transfer + accelerator time: {t_dma * 1e3:.3f} ms")
print("Hardware output shape:", hw_out.shape)
print("Top-left 8x8 of hardware output:")
print(np.rint(hw_out[:8, :8]).astype(int))


--- DMA Streaming 2D Convolution, ap_fixed<32,16> ---
DMA transfer + accelerator time: 5.987 ms
Hardware output shape: (66, 66)
Top-left 8x8 of hardware output:
[[   -9   -26   -50   -74   -98  -122  -146  -170]
 [ -591 -1131 -1618 -1657 -1696 -1735 -1774 -1813]
 [-1554 -2931 -4128 -4173 -4218 -4263 -4308 -4353]
 [-2706 -5043 -7008 -7053 -7098 -7143 -7188 -7233]
 [-1554 -2803 -3744 -3789 -3834 -3879 -3924 -3969]
 [-1170 -2099 -2784 -2829 -2874 -2919 -2964 -3009]
 [-1554 -2931 -4128 -4173 -4218 -4263 -4308 -4353]
 [-2706 -5043 -7008 -7053 -7098 -7143 -7188 -7233]]


## 8. Verify correctness

The hardware output should match the Python reference model. Since this is an `ap_fixed<32,16>` design, the comparison uses a small tolerance of one fixed-point LSB.


In [9]:
diff = hw_out - expected
max_abs_diff = np.max(np.abs(diff))
FIXED_LSB = 1.0 / FIXED_SCALE

print("Difference shape:", diff.shape)
print("Top-left 8x8 of difference:")
print(diff[:8, :8])

print(f"Max absolute difference = {max_abs_diff}")
print(f"Fixed-point LSB         = {FIXED_LSB}")

if max_abs_diff <= FIXED_LSB:
    print("PASS: hardware output matches Python reference within one fixed-point LSB")
else:
    print("FAIL: hardware output differs from Python reference")


Difference shape: (66, 66)
Top-left 8x8 of difference:
[[0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]
Max absolute difference = 0.0
Fixed-point LSB         = 1.52587890625e-05
PASS: hardware output matches Python reference within one fixed-point LSB


## 9. AXI Timer measurement

This section reruns the accelerator and measures elapsed cycles using the AXI Timer IP.


In [10]:
TCSR0 = 0x00
TLR0  = 0x04
TCR0  = 0x08

def timer_start(timer):
    timer.write(TLR0, 0)
    timer.write(TCSR0, 0x020)   # LOAD
    timer.write(TCSR0, 0x080)   # ENT, count up

def timer_stop(timer):
    cycles = timer.read(TCR0)
    timer.write(TCSR0, 0x000)
    return cycles

# Reinitialize buffers
in_buf[:N_INPUT_REAL] = float_to_fixed_u32(image.flatten())
in_buf[N_INPUT_REAL:] = 0
out_buf[:] = 0

write_kernel(conv2d, kernel)

dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)

timer_start(hw_timer)
start_ip(conv2d)

dma.sendchannel.wait()
dma.recvchannel.wait()

cycles = timer_stop(hw_timer)

FCLK_MHZ = 100.0
hw_time_us = cycles / FCLK_MHZ

print(f"HW timer: {cycles} cycles = {hw_time_us:.3f} us @ {FCLK_MHZ:.0f} MHz")


HW timer: 113366 cycles = 1133.660 us @ 100 MHz


# Comparison with Scipy

In [13]:
# ------------------------------------------------------------
# SciPy correlate2d timing baseline for ap_fixed version
# ------------------------------------------------------------
import time
import numpy as np

try:
    from scipy.signal import correlate2d
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False

print("\n--- SciPy correlate2d Timing Baseline ---")

if not SCIPY_AVAILABLE:
    print("SciPy is not available on this PYNQ image.")
    print("Install scipy or use the NumPy sliding-window fallback instead.")
else:
    # Convert fixed-point/raw values back to float for fair numerical comparison.
    # These helper names should already exist in your ap_fixed notebook:
    #   fixed_u32_to_float(...)
    #   float_to_fixed_u32(...)
    #
    # If your image/kernel variables are already float-valued originals,
    # you can use them directly. This version is robust to raw fixed buffers.

    image_float = image.astype(np.float32)
    kernel_float = kernel.astype(np.float32)

    # HLS convention:
    # The kernel is already in sliding-window orientation, so use correlate2d,
    # not convolve2d.
    y_scipy = correlate2d(
        image_float,
        kernel_float,
        mode="full",
        boundary="fill",
        fillvalue=0
    ).astype(np.float32)

    # hw_out_float should be the decoded hardware output from your ap_fixed notebook.
    # If your variable is named differently, replace hw_out_float below.
    max_diff_scipy = np.max(np.abs(hw_out - y_scipy))

    print(f"Max |FPGA ap_fixed output - SciPy correlate2d| = {max_diff_scipy:.8f}")

    # Timing
    N_RUNS = 1000

    # Warm-up
    _ = correlate2d(
        image_float,
        kernel_float,
        mode="full",
        boundary="fill",
        fillvalue=0
    )

    t_start = time.perf_counter()

    for _ in range(N_RUNS):
        y_scipy = correlate2d(
            image_float,
            kernel_float,
            mode="full",
            boundary="fill",
            fillvalue=0
        )

    t_total = time.perf_counter() - t_start
    t_one = t_total / N_RUNS

    print(f"Runs: {N_RUNS}")
    print(f"SciPy correlate2d average time: {t_one * 1e3:.6f} ms")

    print("\n--- FPGA ap_fixed vs SciPy Timing ---")
    print(f"FPGA DMA+accelerator time: {hw_time_us / 1000:.6f} ms")
    print(f"SciPy correlate2d time:    {t_one * 1e3:.6f} ms")

    speedup = (t_one * 1e6) / hw_time_us
    print(f"FPGA speedup vs SciPy:     {speedup:.3f}x")

    # Throughput
    output_pixels = OH * OW
    pixels_per_cycle = output_pixels / cycles
    cycles_per_pixel = cycles / output_pixels
    mpixels_per_sec = pixels_per_cycle * 100e6 / 1e6

    print("\n--- FPGA Throughput ---")
    print(f"Output pixels:             {output_pixels}")
    print(f"Measured cycles:           {cycles}")
    print(f"Pixels/cycle:              {pixels_per_cycle:.6f}")
    print(f"Cycles/output pixel:       {cycles_per_pixel:.3f}")
    print(f"Throughput @ 100 MHz:      {mpixels_per_sec:.3f} MPixels/s")


--- SciPy correlate2d Timing Baseline ---
Max |FPGA ap_fixed output - SciPy correlate2d| = 0.00000000
Runs: 1000
SciPy correlate2d average time: 2.197760 ms

--- FPGA ap_fixed vs SciPy Timing ---
FPGA DMA+accelerator time: 1.133660 ms
SciPy correlate2d time:    2.197760 ms
FPGA speedup vs SciPy:     1.939x

--- FPGA Throughput ---
Output pixels:             4356
Measured cycles:           113366
Pixels/cycle:              0.038424
Cycles/output pixel:       26.025
Throughput @ 100 MHz:      3.842 MPixels/s


## 10. Optional: final verification after timed run


In [11]:
hw_out_timer = fixed_u32_to_float(np.array(out_buf)).reshape(OH, OW)
max_abs_diff_timer = np.max(np.abs(hw_out_timer - expected))

print(f"Max absolute difference after timed run = {max_abs_diff_timer}")
if max_abs_diff_timer <= FIXED_LSB:
    print("PASS: timed-run hardware output matches Python reference within one fixed-point LSB")
else:
    print("FAIL: timed-run hardware output differs from Python reference")


Max absolute difference after timed run = 0.0
PASS: timed-run hardware output matches Python reference within one fixed-point LSB


## 11. Cleanup

Run this cell when finished with the DMA buffers.


In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()
print("DMA buffers freed")
